<a href="https://colab.research.google.com/github/MParvan/ecg-biometrics-bench/blob/main/experiments/load_dataset_Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading datasets

`load_dataset.py` presents seven public ECG datasets through one
interface. It handles the download, the format differences, the
preprocessing, and — most importantly — the split logic that keeps
enrollment and probe data separate.

Every loader exposes the same two methods:

- `load_all_data()` returns everything, for protocols that split
  downstream.
- `load_session(name)` returns one partition, where `name` is
  `'train'` or `'test'`.

In [ ]:
# Clone the framework and install its dependencies.
# On Colab this takes two to three minutes, mostly PyTorch.
!git clone https://github.com/MParvan/ecg-biometrics-bench.git
%cd ecg-biometrics-bench
!pip install -q -r requirements.txt

In [ ]:
from load_dataset import (
    load_ecgid_dataset,
    load_heartprint_dataset,
    load_mitbih_dataset,
)
import numpy as np

## 1. Basic loading

Preprocessing is configured through `preprocessing_config`, a single
mapping that is validated on construction and recorded in the
experiment log. Unknown keys are rejected rather than ignored, so a
typo surfaces immediately.

In [ ]:
loader = load_ecgid_dataset(
    data_split_mode='all-available',
    signal_type='filtered',      # ECG-ID ships raw and filtered channels
    num_beats_to_merge=1,        # one heartbeat per sample
    preprocessing_config={
        'mode': 'beat',                    # segment around R-peaks
        'rpeak_method': 'pantompkins',
        'normalization_method': 'zscore',
    },
)

x, y = loader.load_all_data()
print(f'samples : {x.shape}')
print(f'subjects: {len(np.unique(y))}')

## 2. Temporally separated splits

Randomly splitting heartbeats from one recording inflates results,
because near-identical beats land on both sides. The record-order
regimes avoid this by construction.

`leave-last-out-short-term` enrols on every day-one recording except
the last, and probes the last one. Enrollment always precedes the
probe.

In [ ]:
strict = load_ecgid_dataset(
    data_split_mode='leave-last-out-short-term'
)

x_enroll, y_enroll = strict.load_session('train')
x_probe,  y_probe  = strict.load_session('test')

print(f'enrollment: {x_enroll.shape}')
print(f'probe     : {x_probe.shape}')

You can verify the temporal ordering without training anything. The
audit reuses the same selection code the pipeline runs, so it
describes the partitions that are actually evaluated:

In [ ]:
!python -m scripts.audit_temporal_causality \
  --dataset ecgid \
  --data_split_mode leave-last-out-short-term

## 3. Session-structured datasets

Heartprint and CYBHi provide named sessions, so the split is stated
directly. Subjects missing any requested session are dropped, and
the count is reported.

In [ ]:
heartprint = load_heartprint_dataset(
    data_split_mode='cross-session',
    train_sessions=['session1'],
    enroll_sessions=['session1'],
    probe_sessions=['session2'],
    num_beats_to_merge=3,          # average three beats per sample
    beat_merge_method='average',
    beat_merge_stride=3,           # non-overlapping windows
)

x_train, y_train = heartprint.load_session('train')
x_test,  y_test  = heartprint.load_session('test')
print(f'session 1: {x_train.shape}')
print(f'session 2: {x_test.shape}')

**On `beat_merge_stride`:** merging consecutive beats slides one beat
at a time by default, so neighbouring samples share beats. Setting
the stride equal to the merge width makes them disjoint, which
matters when a random split follows.

## 4. Continuous recordings

MIT-BIH and NSRDB are one long recording per subject rather than
discrete sessions, so partitions are explicit minute windows.

The loader validates that the enrollment coverage does not intersect
the probe coverage, and refuses the configuration if it does.

In [ ]:
mitbih = load_mitbih_dataset(
    leads=['MLII'],
    data_split_mode='custom-split',
    train_parts=[(0, 5)],       # minutes 0-5
    enrol_parts=[(0, 5)],       # the gallery is the training partition
    test_parts=[(25, 30)],      # minutes 25-30
)

print('realised windows:')
for key, value in mitbih.temporal_partition_audit.items():
    if key in ('enrollment_coverage', 'probe_coverage',
               'achieved_separation_minutes'):
        print(f'  {key:<30} {value}')

Try changing `test_parts` to `[(3, 8)]` and re-running the cell. The
loader raises immediately, naming the overlapping minutes, rather
than silently producing an inflated result.